# 01 (SQL) · Crear las tablas del Lakehouse con `CREATE TABLE`

Hace lo mismo que `01_cargar_lakehouse`, pero con SQL: cada tabla se crea con su `CREATE TABLE` y se carga con un `INSERT … SELECT` desde el CSV. Usa uno de los dos notebooks, no los dos.

**Antes de ejecutarlo**

1. Sube los ocho CSV de la carpeta `datos/` del repositorio a **Files** del Lakehouse `Bretema_LH` (sin subcarpeta).
2. Añade `Bretema_LH` como Lakehouse predeterminado del notebook, en el panel **Explorer**.

Cada celda contiene una sola sentencia de Spark SQL. Para cada tabla hay tres pasos:

1. Una **vista temporal** sobre el CSV, con el tipo de cada columna.
2. El **`CREATE TABLE`** de la tabla Delta, con la clave marcada como `NOT NULL`.
3. El **`INSERT`** que copia los datos de la vista a la tabla.

Los importes son `DOUBLE` porque el grafo de la ontología no admite `DECIMAL`. El punto de conexión de análisis SQL del Lakehouse es de solo lectura para las tablas: por eso el `CREATE TABLE` se ejecuta aquí, con Spark, y no allí.

Los datos son ficticios. Banco Brétema no existe.

In [ ]:
%%sql
SET spark.sql.session.timeZone = UTC

## Oficinas

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW csv_oficinas (
    oficina_id STRING, nombre STRING, municipio STRING, provincia STRING, tipo STRING, latitud DOUBLE, longitud DOUBLE
)
USING csv
OPTIONS (path 'Files/oficinas.csv', header 'true', dateFormat 'yyyy-MM-dd', timestampFormat 'yyyy-MM-dd HH:mm:ss', quote '"', escape '"')

In [ ]:
%%sql
CREATE OR REPLACE TABLE oficinas (
    oficina_id STRING NOT NULL,
    nombre STRING,
    municipio STRING,
    provincia STRING,
    tipo STRING,
    latitud DOUBLE,
    longitud DOUBLE
) USING DELTA

In [ ]:
%%sql
INSERT INTO oficinas
SELECT * FROM csv_oficinas

## Gestores

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW csv_gestores (
    gestor_id STRING, nombre STRING, rol STRING, oficina_id STRING
)
USING csv
OPTIONS (path 'Files/gestores.csv', header 'true', dateFormat 'yyyy-MM-dd', timestampFormat 'yyyy-MM-dd HH:mm:ss', quote '"', escape '"')

In [ ]:
%%sql
CREATE OR REPLACE TABLE gestores (
    gestor_id STRING NOT NULL,
    nombre STRING,
    rol STRING,
    oficina_id STRING
) USING DELTA

In [ ]:
%%sql
INSERT INTO gestores
SELECT * FROM csv_gestores

## Clientes

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW csv_clientes (
    cliente_id STRING, nombre STRING, tipo_cliente STRING, segmento STRING, edad INT, municipio STRING, provincia STRING, fecha_alta DATE, oficina_id STRING, gestor_id STRING, riesgo_kyc STRING
)
USING csv
OPTIONS (path 'Files/clientes.csv', header 'true', dateFormat 'yyyy-MM-dd', timestampFormat 'yyyy-MM-dd HH:mm:ss', quote '"', escape '"')

In [ ]:
%%sql
CREATE OR REPLACE TABLE clientes (
    cliente_id STRING NOT NULL,
    nombre STRING,
    tipo_cliente STRING,
    segmento STRING,
    edad INT,
    municipio STRING,
    provincia STRING,
    fecha_alta DATE,
    oficina_id STRING,
    gestor_id STRING,
    riesgo_kyc STRING
) USING DELTA

In [ ]:
%%sql
INSERT INTO clientes
SELECT * FROM csv_clientes

## Cuentas

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW csv_cuentas (
    cuenta_id STRING, cliente_id STRING, tipo_cuenta STRING, fecha_apertura DATE, saldo_eur DOUBLE, estado STRING
)
USING csv
OPTIONS (path 'Files/cuentas.csv', header 'true', dateFormat 'yyyy-MM-dd', timestampFormat 'yyyy-MM-dd HH:mm:ss', quote '"', escape '"')

In [ ]:
%%sql
CREATE OR REPLACE TABLE cuentas (
    cuenta_id STRING NOT NULL,
    cliente_id STRING,
    tipo_cuenta STRING,
    fecha_apertura DATE,
    saldo_eur DOUBLE,
    estado STRING
) USING DELTA

In [ ]:
%%sql
INSERT INTO cuentas
SELECT * FROM csv_cuentas

## Tarjetas

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW csv_tarjetas (
    tarjeta_id STRING, cuenta_id STRING, cliente_id STRING, tipo_tarjeta STRING, tarjeta_enmascarada STRING, fecha_emision DATE, fecha_caducidad DATE, limite_eur DOUBLE, estado STRING, fecha_bloqueo DATE
)
USING csv
OPTIONS (path 'Files/tarjetas.csv', header 'true', dateFormat 'yyyy-MM-dd', timestampFormat 'yyyy-MM-dd HH:mm:ss', quote '"', escape '"')

In [ ]:
%%sql
CREATE OR REPLACE TABLE tarjetas (
    tarjeta_id STRING NOT NULL,
    cuenta_id STRING,
    cliente_id STRING,
    tipo_tarjeta STRING,
    tarjeta_enmascarada STRING,
    fecha_emision DATE,
    fecha_caducidad DATE,
    limite_eur DOUBLE,
    estado STRING,
    fecha_bloqueo DATE
) USING DELTA

In [ ]:
%%sql
INSERT INTO tarjetas
SELECT * FROM csv_tarjetas

## Comercios

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW csv_comercios (
    comercio_id STRING, nombre STRING, categoria STRING, municipio STRING, pais STRING, canal STRING, nivel_riesgo STRING, latitud DOUBLE, longitud DOUBLE
)
USING csv
OPTIONS (path 'Files/comercios.csv', header 'true', dateFormat 'yyyy-MM-dd', timestampFormat 'yyyy-MM-dd HH:mm:ss', quote '"', escape '"')

In [ ]:
%%sql
CREATE OR REPLACE TABLE comercios (
    comercio_id STRING NOT NULL,
    nombre STRING,
    categoria STRING,
    municipio STRING,
    pais STRING,
    canal STRING,
    nivel_riesgo STRING,
    latitud DOUBLE,
    longitud DOUBLE
) USING DELTA

In [ ]:
%%sql
INSERT INTO comercios
SELECT * FROM csv_comercios

## Transacciones

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW csv_transacciones (
    transaccion_id STRING, tarjeta_id STRING, comercio_id STRING, fecha_hora TIMESTAMP, importe_eur DOUBLE, canal STRING, pais STRING, resultado STRING
)
USING csv
OPTIONS (path 'Files/transacciones.csv', header 'true', dateFormat 'yyyy-MM-dd', timestampFormat 'yyyy-MM-dd HH:mm:ss', quote '"', escape '"')

In [ ]:
%%sql
CREATE OR REPLACE TABLE transacciones (
    transaccion_id STRING NOT NULL,
    tarjeta_id STRING,
    comercio_id STRING,
    fecha_hora TIMESTAMP,
    importe_eur DOUBLE,
    canal STRING,
    pais STRING,
    resultado STRING
) USING DELTA

In [ ]:
%%sql
INSERT INTO transacciones
SELECT * FROM csv_transacciones

## Alertas

In [ ]:
%%sql
CREATE OR REPLACE TEMPORARY VIEW csv_alertas (
    alerta_id STRING, transaccion_id STRING, tarjeta_id STRING, cliente_id STRING, fecha_hora TIMESTAMP, tipo_alerta STRING, motivo STRING, puntuacion_riesgo INT, prioridad STRING, estado STRING, resolucion STRING, fecha_resolucion TIMESTAMP
)
USING csv
OPTIONS (path 'Files/alertas.csv', header 'true', dateFormat 'yyyy-MM-dd', timestampFormat 'yyyy-MM-dd HH:mm:ss', quote '"', escape '"')

In [ ]:
%%sql
CREATE OR REPLACE TABLE alertas (
    alerta_id STRING NOT NULL,
    transaccion_id STRING,
    tarjeta_id STRING,
    cliente_id STRING,
    fecha_hora TIMESTAMP,
    tipo_alerta STRING,
    motivo STRING,
    puntuacion_riesgo INT,
    prioridad STRING,
    estado STRING,
    resolucion STRING,
    fecha_resolucion TIMESTAMP
) USING DELTA

In [ ]:
%%sql
INSERT INTO alertas
SELECT * FROM csv_alertas

## Tablas para la Fabric App

La consola de alertas lee el Lakehouse a través de su punto de conexión SQL. Para que no tenga que cruzar seis tablas en cada consulta, se le preparan dos tablas ya cruzadas: `app_alertas` (una fila por alerta, con cliente, oficina, gestor, tarjeta, comercio y el gasto medio de la tarjeta en los 90 días anteriores) y `app_movimientos` (las transacciones con el nombre y la categoría del comercio).

La ontología no usa estas tablas: se enlaza a las tablas del banco.

In [ ]:
%%sql
CREATE OR REPLACE TABLE app_alertas (
    alerta_id STRING NOT NULL,
    fecha_hora TIMESTAMP,
    tipo_alerta STRING,
    motivo STRING,
    puntuacion_riesgo INT,
    prioridad STRING,
    estado STRING,
    resolucion STRING,
    fecha_resolucion TIMESTAMP,
    transaccion_id STRING,
    importe_eur DOUBLE,
    canal STRING,
    pais STRING,
    comercio STRING,
    categoria_comercio STRING,
    municipio_comercio STRING,
    tarjeta_id STRING NOT NULL,
    tipo_tarjeta STRING,
    tarjeta_enmascarada STRING,
    estado_tarjeta STRING,
    cliente_id STRING,
    cliente STRING,
    segmento STRING,
    provincia_cliente STRING,
    oficina STRING,
    gestor STRING,
    gasto_medio_eur DOUBLE
) USING DELTA

In [ ]:
%%sql
INSERT INTO app_alertas
WITH gasto AS (
    SELECT a.alerta_id, AVG(t.importe_eur) AS gasto_medio_eur
    FROM alertas a
    JOIN transacciones t
      ON t.tarjeta_id = a.tarjeta_id
     AND t.fecha_hora <  a.fecha_hora - INTERVAL 3 HOURS
     AND t.fecha_hora >= a.fecha_hora - INTERVAL 90 DAYS
     AND t.resultado = 'Aprobada'
    GROUP BY a.alerta_id
)
SELECT a.alerta_id, a.fecha_hora, a.tipo_alerta, a.motivo, a.puntuacion_riesgo, a.prioridad,
       a.estado, a.resolucion, a.fecha_resolucion,
       t.transaccion_id, t.importe_eur, t.canal, t.pais,
       co.nombre AS comercio, co.categoria AS categoria_comercio, co.municipio AS municipio_comercio,
       ta.tarjeta_id, ta.tipo_tarjeta, ta.tarjeta_enmascarada, ta.estado AS estado_tarjeta,
       c.cliente_id, c.nombre AS cliente, c.segmento, c.provincia AS provincia_cliente,
       o.nombre AS oficina, ge.nombre AS gestor,
       ROUND(gm.gasto_medio_eur, 2) AS gasto_medio_eur
FROM alertas a
JOIN transacciones t ON t.transaccion_id = a.transaccion_id
JOIN comercios co    ON co.comercio_id   = t.comercio_id
JOIN tarjetas ta     ON ta.tarjeta_id    = a.tarjeta_id
JOIN clientes c      ON c.cliente_id     = a.cliente_id
JOIN oficinas o      ON o.oficina_id     = c.oficina_id
JOIN gestores ge     ON ge.gestor_id     = c.gestor_id
LEFT JOIN gasto gm   ON gm.alerta_id     = a.alerta_id

In [ ]:
%%sql
CREATE OR REPLACE TABLE app_movimientos (
    transaccion_id STRING NOT NULL,
    tarjeta_id STRING NOT NULL,
    fecha_hora TIMESTAMP,
    importe_eur DOUBLE,
    canal STRING,
    pais STRING,
    resultado STRING,
    comercio STRING,
    categoria_comercio STRING,
    municipio_comercio STRING,
    riesgo_comercio STRING
) USING DELTA

In [ ]:
%%sql
INSERT INTO app_movimientos
SELECT t.transaccion_id, t.tarjeta_id, t.fecha_hora, t.importe_eur, t.canal, t.pais, t.resultado,
       co.nombre AS comercio, co.categoria AS categoria_comercio, co.municipio AS municipio_comercio,
       co.nivel_riesgo AS riesgo_comercio
FROM transacciones t
JOIN comercios co ON co.comercio_id = t.comercio_id

## Tabla vacía para las decisiones de la app

La ontología enlaza la entidad **Revision** con la tabla `revisiones`. Se crea vacía ahora; el notebook `02_sincronizar_revisiones` la rellena con las decisiones de la consola de alertas.

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS revisiones (
    revision_id STRING NOT NULL,
    alerta_id STRING,
    decision STRING,
    bloquear_tarjeta BOOLEAN,
    comentario STRING,
    analista_email STRING,
    revisada_en TIMESTAMP
) USING DELTA

## Comprobar

Las filas de cada tabla y las cifras que usa el guion de la demo. Todas las filas deben salir con `OK`.

In [ ]:
%%sql
SELECT comprobacion, valor, esperado, CASE WHEN valor = esperado THEN 'OK' ELSE 'REVISAR' END AS resultado
FROM (
    SELECT  1 AS orden, 'Filas en oficinas' AS comprobacion, COUNT(*) AS valor, 24 AS esperado FROM oficinas
    UNION ALL SELECT  2, 'Filas en gestores', COUNT(*), 48 FROM gestores
    UNION ALL SELECT  3, 'Filas en clientes', COUNT(*), 2400 FROM clientes
    UNION ALL SELECT  4, 'Filas en cuentas', COUNT(*), 3194 FROM cuentas
    UNION ALL SELECT  5, 'Filas en tarjetas', COUNT(*), 3236 FROM tarjetas
    UNION ALL SELECT  6, 'Filas en comercios', COUNT(*), 294 FROM comercios
    UNION ALL SELECT  7, 'Filas en transacciones', COUNT(*), 161462 FROM transacciones
    UNION ALL SELECT  8, 'Filas en alertas', COUNT(*), 949 FROM alertas
    UNION ALL SELECT  9, 'Filas en app_alertas', COUNT(*), 949 FROM app_alertas
    UNION ALL SELECT 10, 'Filas en app_movimientos', COUNT(*), 161462 FROM app_movimientos
    UNION ALL SELECT 11, 'Alertas abiertas', COUNT(*), 53 FROM alertas WHERE estado = 'Abierta'
    UNION ALL SELECT 12, 'Tarjetas bloqueadas por fraude', COUNT(*), 37 FROM tarjetas WHERE estado = 'Bloqueada'
    UNION ALL SELECT 13, 'Tarjetas en Ponte Alta del 3 al 11 de octubre', COUNT(DISTINCT t.tarjeta_id), 93
              FROM transacciones t JOIN comercios c ON c.comercio_id = t.comercio_id
              WHERE c.nombre = 'Gasolineira Ponte Alta'
                AND t.fecha_hora BETWEEN '2026-10-03' AND '2026-10-11 23:59:59'
) AS comprobaciones
ORDER BY orden

Las cinco oficinas con más alertas abiertas. Deben ser Vigo Centro (12), Vigo Bouzas (5) y Ourense Centro (4) las tres primeras.

In [ ]:
%%sql
SELECT o.nombre AS oficina, COUNT(*) AS alertas_abiertas
FROM alertas a
JOIN clientes c ON c.cliente_id = a.cliente_id
JOIN oficinas o ON o.oficina_id = c.oficina_id
WHERE a.estado = 'Abierta'
GROUP BY o.nombre
ORDER BY alertas_abiertas DESC
LIMIT 5

**Al terminar**, pulsa **Stop session** para no consumir la capacidad.